Task 1 and 2 solutions

In [ ]:
import qiskit, qiskit_aer, qiskit_ibm_runtime
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.circuit.library import HGate, XGate, YGate, ZGate, CXGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise import pauli_error
from qiskit import generate_preset_pass_manager
from qiskit import QuantumCircuit, transpile

from qff_basic_benchmark import MODEL_NAMES, SHOTS, load_model, draw_model

# The contemporary model is defined by the FakeFez snapshot in qiskit-ibm-runtime 0.50.0.
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__,
      "| qiskit-ibm-runtime", qiskit_ibm_runtime.__version__)
print("shots per circuit:", SHOTS)

In [ ]:
models = {name: load_model(name) for name in MODEL_NAMES}

for name, m in models.items():
    noise = m.error_rates if m.error_rates is not None else f"{m.noise_source} calibration data (Aer)"
    print(f"{name:13s} qubits={m.num_qubits:3d}  couplers={len(m.edges):3d}  "
          f"ports={m.ports}  port distance={m.port_distance()}  gates={m.basis_gates}")
    print(f"{'':13s} noise: {noise}")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

for name in MODEL_NAMES:
    fig = draw_model(models[name])
    display(fig)      # show each figure once
    plt.close(fig)

In [ ]:
results={}
ancilla_ports= {'2016':1, 'contemporary':127, 'hyperbolic':1}
for name, m in models.items():

    passm_noisy = generate_preset_pass_manager(optimization_level=2, backend=m.backend, initial_layout=list(m.ports), seed_transpiler=2026)
    passm_ideal = generate_preset_pass_manager(optimization_level=2, backend=m.ideal_backend, initial_layout=list(m.ports), seed_transpiler=2026)
    new_layout= list(m.ports)+[ancilla_ports[name]]
    passm_prot= generate_preset_pass_manager(optimization_level=2, backend=m.backend, initial_layout=new_layout, seed_transpiler=2026)
    expec_noisy, sigmas_noisy= {}, {}
    expec_ideal, sigmas_ideal = {}, {}
    expec_prot, sigmas_prot, yield_prot={}, {}, {}
    for basis in ['XX', 'YY', 'ZZ']:
        # Generate the specific basis circuit
        bell = create_bell_basis(basis)
        
        tq_noisy=passm_noisy.run(bell)
        tq_ideal=passm_ideal.run(bell)
        bell_prot= create_bell_protected(basis)
        tq_prot=passm_prot.run(bell_prot)
        
        if basis == 'ZZ':
            two_q=sum(1 for inst in tq_noisy.data if inst.operation.num_qubits==2)
            depth=tq_noisy.depth()
            ops=dict(tq_noisy.count_ops())
            depth_prot= tq_prot.depth()
            two_q_prot=sum(1 for inst in tq_prot.data if inst.operation.num_qubits==2)

        counts_noisy=m.backend.run(tq_noisy, shots=SHOTS, seed_simulator=2026).result().get_counts()
        counts_ideal=m.ideal_backend.run(tq_ideal, shots=SHOTS, seed_simulator=2026).result().get_counts()
        counts_prot= m.backend.run(tq_prot, shots= SHOTS, seed_simulator= 2026).result().get_counts()
        expec_noisy[basis],sigmas_noisy[basis]=get_expectation(counts_noisy)
        expec_ideal[basis],sigmas_ideal[basis]=get_expectation(counts_ideal)
        expec_prot[basis], sigmas_prot[basis], yield_prot[basis]= get_protected_expectation(counts_prot)
    mean_yield=np.mean([yield_prot['XX'], yield_prot['YY'], yield_prot['ZZ']])
    F_noisy=(1+expec_noisy['XX']-expec_noisy['YY']+expec_noisy['ZZ'])/4.0
    F_ideal=(1+expec_ideal['XX']-expec_ideal['YY']+expec_ideal['ZZ'])/4.0
    F_prot=(1+expec_prot['XX']-expec_prot['YY']+expec_prot['ZZ'])/4.0
    sigma_final_noisy=0.25*np.sqrt(sigmas_noisy['XX']**2+sigmas_noisy['YY']**2+sigmas_noisy['ZZ']**2)
    sigma_final_ideal=0.25*np.sqrt(sigmas_ideal['XX']**2+sigmas_ideal['YY']**2+sigmas_ideal['ZZ']**2)
    sigma_final_prot=0.25*np.sqrt(sigmas_prot['XX']**2+sigmas_prot['YY']**2+sigmas_prot['ZZ']**2)
    results[name] = {'F_ideal': F_ideal, 'sigma_ideal': sigma_final_ideal,'F_noisy': F_noisy, 'sigma_noisy': sigma_final_noisy, 'F_prot': F_prot, 'sigma_prot': sigma_final_prot, 'yield': mean_yield, 'discarded': SHOTS - int(mean_yield * SHOTS), 'depth': depth, 'two_q': two_q, 'ops': ops, 'depth_prot': depth_prot, 'two_q_prot': two_q_prot}

print("\n--- Final Fidelity Results ---")
for name, data in results.items():
    print(f"[{name.upper()}]")
    print(f"  Cost (Baseline)  : Depth {data['depth']} | {data['two_q']} 2Q Gates")
    print(f"  Cost (Protected) : Depth {data['depth_prot']} | {data['two_q_prot']} 2Q Gates")
    print(f"  Ideal Baseline : F = {data['F_ideal']:.4f} ± {data['sigma_ideal']:.4f} | Verified: {(data['F_ideal']-2*data['sigma_ideal'])>0.5}")
    print(f"  Noisy Baseline : F = {data['F_noisy']:.4f} ± {data['sigma_noisy']:.4f} | Verified: {(data['F_noisy']-2*data['sigma_noisy'])>0.5}")
    print(f"  Noisy + Protect  : F = {data['F_prot']:.4f} ± {data['sigma_prot']:.4f} | Verified: {(data['F_prot']-2*data['sigma_prot'])>0.5}")
    print(f"  Post-Selection   : {data['yield']*100:.1f}% Acceptance ({data['discarded']} shots discarded)")
    print("-" * 75)

In [ ]:
def create_bell_basis(basis):
    circuit = QuantumCircuit(2, 2)
    circuit.h(0)
    circuit.cx(0, 1)
    
    if basis == 'XX':
        circuit.h([0,1])
    elif basis == 'YY':
        circuit.sdg([0,1])
        circuit.h([0,1])
    circuit.measure([0,1], [0,1])
    return circuit

def create_bell_protected(basis):
    circuit= QuantumCircuit(3,3)
    circuit.h(0)
    circuit.cx(0,1)
    circuit.cx(0, 2)
    circuit.cx(1, 2)
    circuit.measure(2, 2)
    if basis == 'XX':
        circuit.h([0,1])
    elif basis == 'YY':
        circuit.sdg([0,1])
        circuit.h([0,1])
    circuit.measure([0,1], [0,1])
    return circuit

def get_protected_expectation(counts):
    accepted_counts = {}
    total_accepted = 0
    for stri, cnt in counts.items():
        if stri[0]=='0':
            total_accepted+=cnt
            data= stri[1:]
            if accepted_counts.get(data):
                accepted_counts[data]+=1
            else:
                accepted_counts[data]=1
    if total_accepted==0:
        return 0.0,0.0,0.0
    yields= total_accepted/SHOTS
    expec= (2*total_accepted-SHOTS)/SHOTS
    sigma= np.sqrt((1-expec**2)/SHOTS)
    return expec, sigma, yields


def get_expectation(counts):
    same= counts.get('00', 0) + counts.get('11', 0)
    diff= counts.get('01', 0) + counts.get('10', 0)
    expec= (same-diff)/SHOTS
    sigma= np.sqrt(max(0,1-expec**2)/SHOTS)
    return expec,sigma


In [ ]:
print("\nTask 4: Core Comparison Table\n")
print("| Processor model | Metric | Ideal baseline | Noisy baseline | Noisy + protection |")
print("| :--- | :--- | :--- | :--- | :--- |")
for name in models.keys():
    d = results[name]
    swaps_base = (d['two_q'] - 1) // 3
    swaps_prot = (d['two_q_prot'] - 3) // 3
    row_name = name.replace('_', ' ').title()
    if name == 'contemporary': row_name = 'Contemporary IBM-style'
    if name == 'hyperbolic': row_name = 'Hyperbolic-inspired'
    if name == '2016': row_name = '2016-inspired'
    print(f"| |{row_name}              | Output quality | F = {d['F_ideal']:.3f} ± {d['sigma_ideal']:.3f} | F = {d['F_noisy']:.3f} ± {d['sigma_noisy']:.3f} | F = {d['F_prot']:.3f} ± {d['sigma_prot']:.3f} |")
    print(f"| | Circuit cost              | Depth {d['depth']}, {d['two_q']} 2Q gates | Depth {d['depth']}, {d['two_q']} 2Q gates | Depth {d['depth_prot']}, {d['two_q_prot']} 2Q gates |")
    print(f"| | Routing                   | {swaps_base} SWAPs | {swaps_base} SWAPs | {swaps_prot} SWAPs |")
    print(f"| | Qubit use                 | 2 physical         | 2 physical         | 3 physical |")
    print(f"| | Post-selection            | N/A.               | N/A                | {d['yield']*100:.1f}% accepted ({d['discarded']} discarded) |")